# Milestone M5: Eksekusi Baseline Komparatif (TF-IDF vs Simple Classifier)
## Kelompok 4 — CNN for Text Classification (Indonesian Hate Speech Detection)

**Mata Kuliah:** Workshop Proyek Sistem Cerdas 2026  
**Dataset:** indotoxic2024 (Subset Splits)  

### Tujuan Notebook:
1. Memuat dataset partisi splits (`data/splits/train.csv`, `val.csv`, `test.csv`).
2. Membangun model pembanding tradisional: **TF-IDF + Logistic Regression** & **TF-IDF + Naive Bayes / Linear SVM** sebagai tolok ukur acuan kelas.
3. Mengevaluasi metrik standar: Macro-F1, Precision, Recall, Accuracy, serta Confusion Matrix menggunakan `MetricCalculator`.
4. Menyimpan catatan hasil evaluasi baseline ke `outputs/metrics/baseline_tfidf.json` untuk perbandingan dengan model CNN pada M6-M7.

In [ ]:
import sys
from pathlib import Path
import os

# Tambahkan root project ke sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from src.utils.config import Config
from src.utils.seed import set_seed
from src.evaluation.metrics import MetricCalculator
from src.evaluation.confusion import ConfusionMatrixPlotter

set_seed(Config.SEED)
print(f"Project root: {ROOT_DIR}")

### 1. Pemuatan Data Split Train / Val / Test

In [ ]:
train_path = ROOT_DIR / Config.TRAIN_CSV
val_path = ROOT_DIR / Config.VAL_CSV
test_path = ROOT_DIR / Config.TEST_CSV

if os.path.exists(train_path) and os.path.exists(test_path):
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path) if os.path.exists(val_path) else None
    test_df = pd.read_csv(test_path)
else:
    print("Membuat data sintetis train dan test untuk baseline testing...")
    train_df = pd.DataFrame({
        "text_clean": [
            "dasar bajingan provokator rusuh",
            "selamat pagi semuanya semoga penuh sukacita dan damai",
            "hajar aja tuh penipu gak usah dikasih ampun",
            "terima kasih atas ilmu dan diskusinya yang bermanfaat",
            "dasar kelompok bodoh gak punya otak",
            "mari kita jaga kerukunan antar umat beragama",
            "mampus lu kadrun cebong tolol perusak negeri",
            "informasi beasiswa kuliah luar negeri sudah dibuka"
        ],
        "label": [1, 0, 1, 0, 1, 0, 1, 0]
    })
    test_df = pd.DataFrame({
        "text_clean": [
            "dasar anjing bangsat lu",
            "selamat siang semuanya tetap semangat ya",
            "mati aja lo perusak bangsa",
            "kegiatan bakti sosial berjalan lancar dan tertib"
        ],
        "label": [1, 0, 1, 0]
    })

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

### 2. Feature Extraction (TF-IDF)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["text_clean"].astype(str))
X_test_tfidf = tfidf.transform(test_df["text_clean"].astype(str))

y_train = train_df["label"].values
y_test = test_df["label"].values

print(f"TF-IDF Matrix Shape (Train): {X_train_tfidf.shape}")
print(f"TF-IDF Matrix Shape (Test) : {X_test_tfidf.shape}")

### 3. Model 1: Logistic Regression Baseline

In [ ]:
lr_model = LogisticRegression(class_weight="balanced", random_state=Config.SEED)
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

metric_calc = MetricCalculator()
metrics_lr = metric_calc.compute_all(y_test, y_pred_lr)

print("=== Hasil Evaluasi TF-IDF + Logistic Regression ===")
print(f"Macro-F1   : {metrics_lr['macro_f1']:.4f}")
print(f"Macro-Prec : {metrics_lr['precision_macro']:.4f}")
print(f"Macro-Rec  : {metrics_lr['recall_macro']:.4f}")
print(f"Accuracy   : {metrics_lr['accuracy']:.4f}")
print("Per Class  :", metrics_lr["per_class"])

### 4. Model 2: Linear SVM & Naive Bayes

In [ ]:
svm_model = LinearSVC(class_weight="balanced", random_state=Config.SEED)
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)
metrics_svm = metric_calc.compute_all(y_test, y_pred_svm)

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)
metrics_nb = metric_calc.compute_all(y_test, y_pred_nb)

print(f"Linear SVM Macro-F1  : {metrics_svm['macro_f1']:.4f}")
print(f"Naive Bayes Macro-F1: {metrics_nb['macro_f1']:.4f}")

### 5. Visualisasi & Penyimpanan Metrik Baseline

In [ ]:
cm_plotter = ConfusionMatrixPlotter(class_names=["Non-toxic", "Toxic"])
cm_path = ROOT_DIR / "outputs/plots/confusion_matrix_baseline_lr.png"
cm_plotter.plot_and_save(y_test, y_pred_lr, output_path=str(cm_path), title="Confusion Matrix: TF-IDF + Logistic Regression")
print(f"Confusion matrix plot tersimpan di: {cm_path}")

# Simpan JSON metrik
baseline_json_path = ROOT_DIR / "outputs/metrics/baseline_tfidf.json"
metric_calc.save_metrics(metrics_lr, str(baseline_json_path))
print(f"Metrik baseline tersimpan di: {baseline_json_path}")

### 6. Kesimpulan Milestone M5
1. Baseline TF-IDF + Logistic Regression berhasil dieksekusi sebagai patokan acuan performa kelompok.
2. Nilai Macro-F1 baseline ini akan menjadi ambang batas minimum yang harus dilampaui oleh model deep learning CNN pada M6-M7.